# Further Experiment 3 — No-shot BERT (masked-LM cloze)

`distilbert-base-uncased` is a **masked language model**. Before any fine-tuning, does it already 'know' sentiment? We probe it with a yes/no cloze — `"is this positive? [MASK] {tweet}"` — and read whether it prefers **yes** or **no** at the `[MASK]`. We try both framings (positive? and negative?) and also a 2-shot version with one example of each sentiment.

**What we noticed:** it is essentially at chance. The base model just has a 'yes' prior (it answers yes most of the time regardless of the tweet), the two framings even disagree, and adding 2 demonstrations does **not** help — a masked LM does not do in-context few-shot learning. This is the flip side of Experiment 5: pretraining alone is a *language* model, not a sentiment classifier; you still need fine-tuning.

## Setup

In [ ]:
!pip install -q datasets transformers scikit-learn

In [ ]:
import re, random, numpy as np, pandas as pd, torch
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForMaskedLM
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
def clean_tweet(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # strip URLs
    text = re.sub(r'@\w+', '@user', text)                  # @mentions -> @user
    text = re.sub(r'\s+', ' ', text).strip()               # normalize whitespace
    return text

## Same 1,000-tweet evaluation set

In [ ]:
ds = load_dataset('contemmcm/sentiment140')['complete']
neg = ds.filter(lambda x: x['label'] == 0).shuffle(seed=SEED).select(range(500))
pos = ds.filter(lambda x: x['label'] == 1).shuffle(seed=SEED).select(range(500))
eval_ds = concatenate_datasets([neg, pos]).shuffle(seed=SEED)
texts = [clean_tweet(t) for t in eval_ds['text']]
labels = np.array(eval_ds['label'])

## Load the base (un-fine-tuned) DistilBERT as a masked LM

In [ ]:
tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
mlm = AutoModelForMaskedLM.from_pretrained('distilbert-base-uncased').to(device).eval()
MASK = tok.mask_token
YES, NO = tok.convert_tokens_to_ids('yes'), tok.convert_tokens_to_ids('no')

## Read P(yes) vs P(no) at the [MASK]

In [ ]:
@torch.no_grad()
def yes_over_no(prompts, B=64):
    out = []
    for i in range(0, len(prompts), B):
        enc = tok(prompts[i:i+B], truncation=True, padding=True, max_length=64,
                  return_tensors='pt').to(device)
        logits = mlm(**enc).logits
        mask = enc.input_ids == tok.mask_token_id
        for b in range(logits.shape[0]):
            pos = mask[b].nonzero(as_tuple=True)[0][-1]
            out.append(bool(logits[b, pos, YES] > logits[b, pos, NO]))
    return np.array(out)

def report(name, preds):
    acc = accuracy_score(labels, preds); tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    print(f'{name:24s} acc={acc:.4f} pos_rec={tp/(tp+fn):.3f} neg_rec={tn/(tn+fp):.3f}')

## Zero-shot, two framings

In [ ]:
# 'positive?' : yes -> positive(1).   'negative?' : yes -> negative(0).
pos_yes = yes_over_no([f'is this positive? {MASK} {t}' for t in texts])
neg_yes = yes_over_no([f'is this negative? {MASK} {t}' for t in texts])
report('zero-shot positive?', pos_yes.astype(int))
report('zero-shot negative?', (~neg_yes).astype(int))
print(f"says 'yes' to positive? {pos_yes.mean()*100:.0f}% | to negative? {neg_yes.mean()*100:.0f}%")

## 2-shot: prepend one positive and one negative demonstration

(Clear, hand-written examples — not from the test set.)

In [ ]:
POS_DEMO = 'just had the best pancakes ever, going to be a great day'
NEG_DEMO = 'my internet has been out for 3 hours, so frustrating'
def shot(q, pos_ans, neg_ans, t):
    return (f'{q} {pos_ans} {POS_DEMO} . {q} {neg_ans} {NEG_DEMO} . {q} {MASK} {t}')
pos_yes2 = yes_over_no([shot('is this positive?', 'yes', 'no', t) for t in texts])
neg_yes2 = yes_over_no([shot('is this negative?', 'no', 'yes', t) for t in texts])
report('2-shot positive?', pos_yes2.astype(int))
report('2-shot negative?', (~neg_yes2).astype(int))
print(f"says 'yes' to positive? {pos_yes2.mean()*100:.0f}% | to negative? {neg_yes2.mean()*100:.0f}%")

## What we found

| Framing | Zero-shot | + 2-shot |
|---|---|---|
| `is this positive?` | 0.559 | 0.554 |
| `is this negative?` | 0.458 | 0.456 |
| says 'yes' to pos? / neg? | 68% / 63% | 74% / 73% |

- The base model is **at/below chance** — it just answers 'yes' most of the time, and the two framings disagree (real understanding wouldn't).
- **2-shot demonstrations don't help** — a masked LM doesn't do in-context learning the way an autoregressive LLM does; it just answers 'yes' even more.
- Compare to the **fine-tuned** DistilBERT (0.83): pretraining gives a language model, *fine-tuning* turns it into a classifier.